# MiniGPT Inference & Dialogue

This notebook loads the saved checkpoint (`mini_gpt.pt`) and allows you to chat or ask questions **instantly without retraining**.

### How it Works:
1. Loads the weights, character vocabulary, and model configuration from `mini_gpt.pt`.
2. Formats your query into the conversation structure:
   ```text
   User: <your question>
   Assistant: 
   ```
3. Generates tokens autoregressively until it encounters the `<|end|>` or `\nUser:` stop delimiter.
4. Extracts and displays the clean answer.


## 1. Imports & Hardware Accelerator
Detect Apple Silicon (`MPS`), Nvidia (`CUDA`), or CPU.


In [ ]:
import math
from pathlib import Path
import torch
import torch.nn as nn
from torch.nn import functional as F

if torch.backends.mps.is_available():
    device = "mps"
elif torch.cuda.is_available():
    device = "cuda"
else:
    device = "cpu"

print(f"[*] Hardware accelerator: {device.upper()}")

## 2. Transformer Architecture Definition
Self-contained model classes matching the saved checkpoint.


In [ ]:
class CausalSelfAttention(nn.Module):
    def __init__(self, n_embd: int, n_head: int, block_size: int, dropout: float = 0.1):
        super().__init__()
        assert n_embd % n_head == 0, "n_embd must be divisible by n_head"
        self.n_head = n_head
        self.head_dim = n_embd // n_head

        self.c_attn = nn.Linear(n_embd, 3 * n_embd, bias=False)
        self.c_proj = nn.Linear(n_embd, n_embd, bias=False)
        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)

        self.register_buffer(
            "bias",
            torch.tril(torch.ones(block_size, block_size)).view(
                1, 1, block_size, block_size
            ),
        )

    def forward(self, x):
        B, T, C = x.size()
        qkv = self.c_attn(x)
        q, k, v = qkv.chunk(3, dim=-1)

        q = q.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_head, self.head_dim).transpose(1, 2)

        att = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(self.head_dim))
        att = att.masked_fill(self.bias[:, :, :T, :T] == 0, float("-inf"))
        att = F.softmax(att, dim=-1)
        att = self.attn_dropout(att)

        y = att @ v
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_dropout(self.c_proj(y))


class MLP(nn.Module):
    def __init__(self, n_embd: int, dropout: float = 0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.GELU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)


class Block(nn.Module):
    def __init__(self, n_embd: int, n_head: int, block_size: int, dropout: float = 0.1):
        super().__init__()
        self.ln_1 = nn.LayerNorm(n_embd)
        self.attn = CausalSelfAttention(n_embd, n_head, block_size, dropout)
        self.ln_2 = nn.LayerNorm(n_embd)
        self.mlp = MLP(n_embd, dropout)

    def forward(self, x):
        x = x + self.attn(self.ln_1(x))
        x = x + self.mlp(self.ln_2(x))
        return x


class MiniGPT(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        n_embd: int = 256,
        n_head: int = 8,
        n_layer: int = 6,
        block_size: int = 256,
        dropout: float = 0.1,
    ):
        super().__init__()
        self.block_size = block_size
        self.vocab_size = vocab_size

        self.tok_emb = nn.Embedding(vocab_size, n_embd)
        self.pos_emb = nn.Embedding(block_size, n_embd)
        self.drop = nn.Dropout(dropout)

        self.blocks = nn.Sequential(
            *[Block(n_embd, n_head, block_size, dropout) for _ in range(n_layer)]
        )
        self.ln_f = nn.LayerNorm(n_embd)
        self.head = nn.Linear(n_embd, vocab_size, bias=False)

        # Weight tying
        self.tok_emb.weight = self.head.weight

    def forward(self, idx, targets=None):
        B, T = idx.size()
        tok_embeddings = self.tok_emb(idx)
        pos = torch.arange(0, T, dtype=torch.long, device=idx.device)
        pos_embeddings = self.pos_emb(pos)
        x = self.drop(tok_embeddings + pos_embeddings)

        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.head(x)
        return logits, None

## 3. Load Checkpoint
Load the trained weights, configuration parameters, and vocabulary from `mini_gpt.pt`.


In [ ]:
# Resolve checkpoint path
ckpt_path = Path("mini_gpt.pt")
if not ckpt_path.is_file():
    ckpt_path = Path("gemini/mini_gpt.pt")

if not ckpt_path.is_file():
    raise FileNotFoundError(f"Checkpoint '{ckpt_path}' not found. Please train first via train.ipynb!")

checkpoint = torch.load(ckpt_path, map_location=device)

chars = checkpoint["chars"]
config = checkpoint["config"]
is_qa_model = checkpoint.get("is_qa_model", True)

stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}
encode = lambda s: [stoi[c] for c in s if c in stoi]
decode = lambda l: "".join([itos[i] for i in l])

model = MiniGPT(**config).to(device)
model.load_state_dict(checkpoint["model_state"])
model.eval()

print(f"[✓] Successfully loaded model from '{ckpt_path}'.")
print(f"[*] Architecture: {config['n_layer']} layers, {config['n_head']} heads, {config['n_embd']} embed dim.")
print(f"[*] Vocabulary size: {config['vocab_size']} unique characters.")

## 4. Query & Generation Function
Format the input with conversation tags and generate answers until a stop delimiter is reached.


In [ ]:
def ask(query: str, max_new_tokens: int = 200, temperature: float = 0.5, top_k: int = 40):
    """Ask MiniGPT a question and return the assistant's answer."""
    prompt = f"User: {query.strip()}\nAssistant: "
    encoded = encode(prompt)
    if not encoded:
        context = torch.zeros((1, 1), dtype=torch.long, device=device)
    else:
        context = torch.tensor([encoded], dtype=torch.long, device=device)

    stop_indicators = ["<|end|>", "\nUser:", "User:"]
    generated_text = ""

    with torch.no_grad():
        for _ in range(max_new_tokens):
            idx_cond = context[:, -model.block_size :]
            logits, _ = model(idx_cond)
            logits = logits[:, -1, :] / max(temperature, 1e-5)

            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = -float("Inf")

            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)

            context = torch.cat((context, idx_next), dim=1)
            next_char = decode([idx_next.item()])
            generated_text += next_char

            if any(stop_str in generated_text for stop_str in stop_indicators):
                break

    # Clean up stop delimiters
    for stop_str in stop_indicators:
        if stop_str in generated_text:
            generated_text = generated_text.split(stop_str)[0]

    return generated_text.strip()

## 5. Test Questions
Try asking multiple questions across different categories.


In [ ]:
questions = [
    "What is the capital of France?",
    "What is Python?",
    "What is 2 + 2?",
    "Why is the sky blue?",
    "Who wrote Romeo and Juliet?",
    "What is an LLM?",
]

for q in questions:
    ans = ask(q)
    print(f"Q: {q}")
    print(f"A: {ans}\n")

## 6. Try Your Own Question
Enter any custom question in `my_question` below and run the cell!


In [ ]:
my_question = "What is the largest planet in our solar system?"

response = ask(my_question)
print(f"You : {my_question}")
print(f"MiniGPT: {response}")